# Train Mnx: tools, images, search, code and answers

This notebook fine-tunes **Qwen2.5-3B-Instruct** into your Mnx model, using data that matches exactly how the Mnx app talks to the model (`<think>`, `<tool_call>`, `<tool_response>`). Then it **measures** the result on held-out test cases and exports GGUF files for your phone.

**Before you start:** Runtime → Change runtime type → **T4 GPU** (free) or better.

Steps: 1) setup → 2) make data → 3) train (~1–2 h on a T4) → 4) export GGUF → 5) evaluate (target 98%) → 6) upload.

In [ ]:
# 1. Check the GPU and install Unsloth
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q unsloth

## 1b. Get the Mnx code
If the repo is private, add a GitHub token as a Colab secret named `GITHUB_TOKEN` (🔑 icon on the left). Add `HF_TOKEN` (write access) to upload the trained model at the end.

In [ ]:
import os, subprocess
try:
    from google.colab import userdata
    gh = userdata.get("GITHUB_TOKEN")
except Exception:
    gh = None
url = f"https://{gh}@github.com/Tharunmaks/Mnx" if gh else "https://github.com/Tharunmaks/Mnx"
if not os.path.exists("Mnx"):
    subprocess.run(["git", "clone", "-q", "-b", "claude/mnx-ai-web-interface-f75vu6", url, "Mnx"], check=True)
!cd Mnx && git log --oneline -1

## 2. Make the training data
3,000 conversations ≈ 1–1.5 hours of training on a T4. More data helps a bit more but takes longer.
The test set uses cities, topics and tasks the model never sees in training.

**Optional:** to keep what your old fine-tune learned, upload its dataset as `extra.jsonl` (one `{"messages": [...]}` per line) and it will be mixed in.

In [ ]:
TRAIN_SIZE = 3000
!cd Mnx && python3 training/generate.py --train {TRAIN_SIZE} --test 400

## 3. Train (LoRA)

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from datasets import load_dataset, concatenate_datasets
from trl import SFTTrainer, SFTConfig
import os

MAX_LEN = 4096
model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/Qwen2.5-3B-Instruct", max_seq_length=MAX_LEN, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=32, lora_alpha=32, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407)
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

data = load_dataset("json", data_files="Mnx/training/data/train.jsonl", split="train")
if os.path.exists("extra.jsonl"):
    data = concatenate_datasets([data.select_columns(["messages"]), load_dataset("json", data_files="extra.jsonl", split="train").select_columns(["messages"])])
data = data.map(lambda ex: {"text": tokenizer.apply_chat_template(ex["messages"], tokenize=False)})
data = data.shuffle(seed=3407)
print(data[0]["text"][-800:])

In [ ]:
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=data,
    args=SFTConfig(
        dataset_text_field="text", max_seq_length=MAX_LEN, packing=False,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        num_train_epochs=2, learning_rate=2e-4, warmup_steps=20,
        lr_scheduler_type="cosine", optim="adamw_8bit", weight_decay=0.01,
        logging_steps=10, save_steps=200, save_total_limit=2,
        output_dir="outputs", seed=3407, report_to="none"))
# Learn only the assistant's turns (thinking, tool calls, answers), not the
# system prompt, user messages or tool results.
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
trainer.train()  # if Colab disconnects, re-run setup cells then: trainer.train(resume_from_checkpoint=True)

## 3b. Quick check

In [ ]:
FastLanguageModel.for_inference(model)
import json
case = json.loads(open("Mnx/training/data/test.jsonl").readline())
ids = tokenizer.apply_chat_template(case["messages"], add_generation_prompt=True, return_tensors="pt").to("cuda")
out = model.generate(input_ids=ids, max_new_tokens=300, temperature=0.7, top_p=0.8, top_k=20)
print("USER:", case["messages"][1]["content"])
print(tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=False))

## 4. Export GGUF for the phone
- **Q4_K_M**: best quality for its size (same as your current model)
- **Q4_0**: recent llama.cpp builds run this faster on most phone CPUs (ARM). Try both on your phone.

In [ ]:
model.save_pretrained_gguf("mnx-gguf", tokenizer, quantization_method=["q4_k_m", "q4_0"])
import glob, shutil
for f in glob.glob("mnx-gguf*/*.gguf") + glob.glob("mnx-gguf*.gguf"):
    low = f.lower()
    if "q4_k_m" in low: shutil.copy(f, "mnx-q4_k_m.gguf")
    elif "q4_0" in low: shutil.copy(f, "mnx-q4_0.gguf")
!ls -lh mnx-*.gguf

## 5. Measure accuracy (target 98%)
Runs every held-out case through Mnx's real tool loop (including the permission step for code) with llama.cpp on the GPU.
This builds llama.cpp once (~5–8 minutes).

In [ ]:
!curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null && apt-get install -y -qq nodejs > /dev/null && node -v
!cd Mnx && npm install --omit=optional --silent
if not os.path.exists("llama.cpp/build/bin/llama-server"):
    !git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
    !cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=ON -DLLAMA_CURL=OFF > /dev/null
    !cmake --build llama.cpp/build --config Release -j 4 --target llama-server > /dev/null

In [ ]:
!cd Mnx && MNX_LLAMA_SERVER_BIN=../llama.cpp/build/bin/llama-server MNX_GPU_LAYERS=99 node training/eval.mjs --model ../mnx-q4_k_m.gguf --target 98

If a category is below 98%, look at the failures printed above (also saved in `Mnx/training/data/eval-report.json`), then:
- train longer (`num_train_epochs=3`) or with more data (`TRAIN_SIZE = 6000`),
- or tell Claude which failures you see so the generator can be improved.

## 6. Upload to Hugging Face

In [ ]:
from huggingface_hub import HfApi
from google.colab import userdata
api = HfApi(token=userdata.get("HF_TOKEN"))
repo = "tharunmakes/mnx-qwen2.5-3b-gguf"
api.create_repo(repo, private=True, exist_ok=True)
for f in ["mnx-q4_k_m.gguf", "mnx-q4_0.gguf"]:
    api.upload_file(path_or_fileobj=f, path_in_repo=f, repo_id=repo)
print("Uploaded. On the phone: HF_TOKEN=hf_... npm run get-model")